In [ ]:
import scipy.io as sio
import numpy as np
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

In [ ]:
# -------------------------------
# 1. 数据加载与预处理
# -------------------------------
# 假设.mat文件名为 'your_data.mat'，数据变量名为 'data'
mat_contents = sio.loadmat('../data/svm/train3.mat')  # 替换为实际文件路径
# print(mat_contents)
data = mat_contents['train_3']  # 根据实际文件中的变量名调整

# 数据中前128列为特征，最后1列为标签（原始标签取值为 1~5）
X = data[:, :128]  # 特征矩阵，尺寸为 (N, d) 其中 d=128
y = data[:, 128].flatten()  # 标签向量，尺寸为 (N, )
# 为方便运算，将标签调整到 0~K-1 （K=5）
y = y - 1

# 获取数据维度
num_samples, d = X.shape
K = len(np.unique(y))

# 参数初始化与超参数设置
np.random.seed(42)
W = 0.001 * np.random.randn(K, d)    # 权重矩阵，尺寸为 (K, d),其中 K=5,d=128
b = np.zeros(K)                      # 偏置向量，尺寸为 (K,)


params = sio.loadmat('../data/svm/multi_svm_params.mat')
Wb_loaded = params['Wb']  # 加载后的矩阵大小为 (K, d+1)

# 2. 假设特征维度 d 已知（例如 d=128）
d = 128

# 3. 拆分参数矩阵
W = Wb_loaded[:, :d]      # 前 d 列为权重矩阵 (K, d)
b = Wb_loaded[:, d].flatten()   # 最后一列为偏置向量 (K,)


# 超参数
learning_rate = 1e-3
lambda_reg = 1e-4
num_iters = 10000
batch_size = 200


In [ ]:
# -------------------------------
# 2. 多分类 SVM（Crammer–Singer形式）问题推导与训练
# -------------------------------
# 对于每个输入 x_i，各类别得分为:
#       s_k = w_k^T x_i + b_k,
# 其中 W 为大小 (K, d) 的权重矩阵，b 为大小 (K,) 的偏置向量。
#
# 损失函数定义为：
#       L = (1/N) * sum_{i=1}^N [sum_{j≠y_i} max(0, 1 - (s_{y_i} - s_j))] + (λ/2) * ||W||^2
#
# 对于每个样本 i，其 margin 定义为：
#       margin_j = s_j - s_{y_i} + 1,  (j ≠ y_i)
# 对于正确类别则不计算 loss（可记为 0）。
#
# 子梯度计算：
# 对于 j ≠ y_i, 当 margin_j > 0，有：
#       ∂L/∂w_j += x_i,   ∂L/∂b_j += 1;
# 对于 j = y_i，有：
#       ∂L/∂w_{y_i} -= (number of positive margins for sample i) * x_i,
#       ∂L/∂b_{y_i} -= (number of positive margins for sample i).
#
# 加上正则化项 (λ/2)*||W||^2 后，
#       ∂L/∂W = computed_gradient + λ*W.
#
# 我们采用 mini-batch 子梯度下降来训练模型。

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize

print(X.shape)
from util import plot_v_cond,plot_cond,show_crossbar,DataLoader
# plot_cond(X,vmax=2)

# plt.hist(X.flatten())
# plt.show()

# print(np.sum(X>1-1e-6))
# print(np.sum(X))
# for i in range(15000):
#     print(X[i])

# sum0=0
# sum1=0
# sum2=0
# for i in X.flatten():
#     if i>-1e-6 and i<1e-6:
#         sum0+=1
#     if i>1-1e-6 and i<1+1e-6:
#         sum1+=1
#     if i>2-1e-6 and i<2+1e-6:
#         sum2+=1
# print(sum0,sum1,sum2)

# 输入只有0,1,2

In [ ]:
loss_history = []

for it in range(num_iters):
    # 随机采样 mini-batch
    batch_idx = np.random.choice(num_samples, batch_size, replace=False)
    X_batch = X[batch_idx]
    y_batch = y[batch_idx]

    # 计算得分矩阵，尺寸 (batch_size, K)
    scores = np.dot(X_batch, W.T) + b

    # 取出每个样本的正确类别得分，尺寸 (batch_size, 1)
    correct_scores = scores[np.arange(batch_size), y_batch].reshape(-1, 1)

    # 计算 margin = s_j - s_{y_i} + 1
    margins = scores - correct_scores + 1
    margins[np.arange(batch_size), y_batch] = 0  # 正确类别不计算 loss

    # 只保留正 margin
    margins = np.maximum(0, margins)

    # 计算损失：平均损失 + 正则化项
    loss = np.sum(margins) / batch_size + 0.5 * lambda_reg * np.sum(W * W)
    loss_history.append(loss)

    # -------------------------------
    # 计算梯度（子梯度）
    # -------------------------------
    # 构造指示矩阵，若 margin > 0 则记为 1
    indicator = (margins > 0).astype(float)
    # 每个样本计算正 margin 的数量
    row_sum = np.sum(indicator, axis=1)  # (batch_size,)
    # 对正确类别的梯度修正
    indicator[np.arange(batch_size), y_batch] = -row_sum

    # 计算梯度：W 的梯度 (K, d)，b 的梯度 (K,)
    dW = np.dot(indicator.T, X_batch) / batch_size + lambda_reg * W
    db = np.sum(indicator, axis=0) / batch_size

    # 参数更新
    W -= learning_rate * dW
    b -= learning_rate * db

    if it % 100 == 0:
        scores_all = np.dot(X, W.T) + b  # 整个训练集的得分，尺寸 (num_samples, K)
        y_pred = np.argmax(scores_all, axis=1)  # 预测的类别（0~K-1）
        acc = np.mean(y_pred == y)

        # 输出当前迭代的损失和预测准确率
        print(f"Iter {it + 1}/{num_iters}: Loss = {loss:.4f}, Accuracy = {acc * 100:.2f}%")
        # print(f"Iteration {it}/{num_iters}, loss = {loss:.4f}")


In [ ]:
# -------------------------------
# 3. 保存 SVM 参数矩阵
# -------------------------------
# 合并 W 和 b 得到一个 K×(d+1) 的矩阵：
# 前 d 列为权重，最后一列为偏置
Wb = np.hstack((W, b.reshape(K, 1)))
print("最终模型参数矩阵的尺寸 (K x [d+1]) =", Wb.shape)

# 保存参数矩阵到 .mat 文件
sio.savemat('multi_svm_params.mat', {'Wb': Wb})
print("训练得到的 SVM 参数矩阵已保存到 'multi_svm_params.mat' 文件中。")

# -------------------------------
# 4. 读入参数矩阵并进行预测
# -------------------------------
# 假设我们需要对新数据 X_new (形状: (N_new, d)) 进行预测
# 此处举个例子，